In [2]:
import numpy as np
import pandas as pd
pd.options.mode.copy_on_write = True # do this until pandas 3.0 is released
import plotly.express as px


def rename_columns(df):
    df.columns = [col.split('#')[-1].strip() if '#' in col else col.strip() for col in df.columns]
    return df

# 1) is it centered?
I ran the centering code called centerCalibGui.py, but seems like it didn't do a great job.

it says that the center coordinates are 31.04499272827137, and 24.73

In [3]:
df1 = rename_columns(pd.read_csv("1centerQ.csv", comment = ";",  parse_dates=[' pd.Timestamp.now #now','queueT'])) #parse dates for convenience only

phi = np.deg2rad(df1['magnet.phi'])
elev = np.deg2rad(df1['magnet.theta'])   # theta = elevation from horizon
B = df1['magnet.field']

df1['magX'] = B * np.cos(elev) * np.cos(phi)
df1['magY'] = B * np.cos(elev) * np.sin(phi)
df1['magZ'] = B * np.sin(elev)
px.scatter(df1, "magX", "Bx").show()
px.scatter(df1, "magY", "By").show()
px.scatter(df1, "magZ", "Bz").show()

# 2) How far off are we?

## 2a) low resolution scan, with no wait -> bad positioning

In [29]:
df2 = rename_columns(pd.read_csv("2MagCal.csv", comment = ";",  parse_dates=[' pd.Timestamp.now #now','queueT'])) #parse dates for convenience only
df2a= df2.query("purpose == 'XYscan'")

df2a.query("Mphi in [-90,0,90]", inplace=True) #when phi moving, it's bad
px.scatter(df2a,"now",["Mphi"]).show() #showing when phi is moving


px.density_heatmap( df2a, x="Mx", y="My", z="Bz", histfunc="avg").show()
px.scatter( df2a, x="Mx", y="My", color="Bz", facet_col="Mphi",color_continuous_scale="RdBu",
    color_continuous_midpoint=0,).show()

# px.scatter(
#     df2a,
#     x="Mx",
#     y="My",
#     color="Bz",
#     facet_col="Mphi",
#     color_continuous_scale="RdBu",
#     color_continuous_midpoint=0,
# ).show()

px.line(df2a,"now",["Mx","My"]).show()




## 2b better resolution raster scan

In [41]:
df2 = rename_columns(pd.read_csv("2MagCal.csv", comment = ";",  parse_dates=[' pd.Timestamp.now #now','queueT'])) #parse dates for convenience only
df2b= df2.query("purpose == 'XYscan withWait'")

df2b.Mphi = df2b.Mphi.round(2) #rounding phi to show the same plot as df2a, but with wait time

px.scatter( df2b, x="Mx", y="My", color="Bz", facet_col="Mphi",color_continuous_scale="RdBu",
    color_continuous_midpoint=0,title="real bX").show()

px.scatter( df2b, x="Mx", y="My", color="Bx", facet_col="Mphi",color_continuous_scale="RdBu",
    color_continuous_midpoint=0,title="real bY").show()

px.scatter( df2b, x="Mx", y="My", color="By", facet_col="Mphi",color_continuous_scale="RdBu",
    color_continuous_midpoint=0, title="real bZ").show()

### finding the zero crossings by eye.

It says the coordinates are 16.8844, 25.3405

In [63]:
Mx0Guess = 16.8844
My0Guess = 25.3405

uniq = np.sort(df2b['Mx'].unique())
closest_Mx = uniq[np.abs(uniq - Mx0Guess).argmin()]
print(f"Closest Mx: {closest_Mx} VERSUS GUESS: {Mx0Guess}")

uniq = np.sort(df2b['My'].unique())
closest_My = uniq[np.abs(uniq - My0Guess).argmin()]
print(f"Closest My: {closest_My} VERSUS GUESS: {My0Guess}")

px.line(df2b.query("My==@closest_My"), "Mx", "By", color="Mphi").add_vline(x=Mx0Guess).show()

px.line(df2b.query("Mx==@closest_Mx"), "My", "By", color="Mphi").add_vline(x=My0Guess).show()


Closest Mx: 16.8182 VERSUS GUESS: 16.8844
Closest My: 25.303 VERSUS GUESS: 25.3405


In [31]:
df2.purpose.unique()

array(['XYscan', 'XYscan withWait'], dtype=object)